In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory  import ConversationBufferMemory,ConversationBufferWindowMemory,ConversationSummaryBufferMemory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

model = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5.1",
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{message}"),
    ]
)

buffer_memory = ConversationBufferMemory(return_messages=True)

wd_memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=3,
)

sum_memory = ConversationSummaryBufferMemory(
    llm=model,
    max_token_limit=100,
    return_messages=True,
)

def load_memory(_):
    return buffer_memory.load_memory_variables({})["history"]

chain = RunnablePassthrough.assign(history=load_memory) | prompt | model


response = chain.invoke(inputs)

c:\GDC\gdc.ojt\labs\week1\practice\cuongld\full-stack-gpt-hw\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
C:\Users\LG CNS\AppData\Local\Temp\ipykernel_28288\2261389016.py:20: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True)


In [ ]:
# 10-turn conversation test for memory
test_conversations = [
    {"input": "Hi, my name is Cuong.", "output": "Nice to meet you, Cuong!"},
    {"input": "I live in Hanoi.", "output": "Great, you live in Hanoi."},
    {"input": "My favorite language is Python.", "output": "Awesome, Python is your favorite language."},
    {"input": "I work as a backend developer.", "output": "Got it, you are a backend developer."},
    {"input": "I enjoy building APIs with FastAPI.", "output": "Cool, you like building APIs with FastAPI."},
    {"input": "My favorite hobby is running.", "output": "Nice, running is your favorite hobby."},
    {"input": "I ran 5km yesterday.", "output": "Great job! You ran 5km yesterday."},
    {"input": "I also like reading AI newsletters.", "output": "Noted, you also enjoy reading AI newsletters."},
    {"input": "Can you summarize what you know about me so far?", "output": "You are Cuong, live in Hanoi, love Python, work backend, and enjoy running."},
    {"input": "What city do I live in and what is my favorite language?", "output": "You live in Hanoi and your favorite language is Python."},
]

# Clear all memories before test
buffer_memory.clear()
wd_memory.clear()
sum_memory.clear()

def add_message(input, output):
    buffer_memory.save_context({"input": input}, {"output": output})
    wd_memory.save_context({"input": input}, {"output": output})
    sum_memory.save_context({"input": input}, {"output": output})

for turn, message in enumerate(test_conversations, start=1):
    user_text = message["input"]
    ai_text = message["output"]
    add_message(user_text, ai_text)
    print(f"Turn {turn}")
    print(f"User: {user_text}")
    print(f"AI: {ai_text}")

history = buffer_memory.load_memory_variables({})["history"]
print(f"Total messages in buffer_memory: {len(history)}")